# TF-IDF Quick Test
Run TF-IDF retrieval on 5 queries from queries.csv.

In [14]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path


def resolve_project_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd / "traditional-retrieval"]
    for path in candidates:
        if (path / "data").exists() and (path / "source").exists():
            return path
    raise FileNotFoundError("Could not locate the traditional-retrieval project directory.")

project_dir = resolve_project_dir()
print(f"Project directory resolved to: {project_dir}")
data_dir = project_dir / "data"
utils_dir = project_dir / "source" / "utils"

for path in (project_dir, utils_dir):
    if path not in sys.path:
        sys.path.append(path)

from source.utils.textPreprocessing import TextPreprocessing
from source.utils.inverted_index import InvertedIndex
from source.utils.Tf_Idf_retriever import TfIDF

Project directory resolved to: C:\Itoni\CS419\final-project\traditional-retrieval


In [9]:
# Load data
documents_path = data_dir / "documents.csv"
queries_path = data_dir / "queries.csv"

documents_df = pd.read_csv(documents_path)
queries_df = pd.read_csv(queries_path)

# Preprocess documents
tp = TextPreprocessing()
processed_docs = tp.process_dataframe(documents_df)
vocab = tp.get_vocab()

# Build inverted index
index_builder = InvertedIndex(processed_docs, vocab)
index_builder._build()

# Initialize TF-IDF retriever
searcher = TfIDF(index_builder, processed_docs)

In [10]:
# Run TF-IDF search on 5 queries and show documents
sample_queries = queries_df.head(5)

top_k = 5
doc_map = documents_df.set_index("id")["content"].to_dict()
results = []
for _, row in sample_queries.iterrows():
    query_id = row["id"]
    query_text = row["content"]
    ranked = searcher.search(query_text, top_k=top_k)
    enriched = []
    for doc_id, score in ranked:
        enriched.append({
            "doc_id": doc_id,
            "score": score,
            "content": doc_map.get(doc_id, "")
        })
    results.append({"query_id": query_id, "query": query_text, "results": enriched})

results

[{'query_id': 1,
  'query': ' what similarity laws must be obeyed when constructing aeroelastic models of heated high speed aircraft .',
  'results': [{'doc_id': 184,
    'score': np.float64(0.2590562641302824),
    'content': 'scale models for thermo aeroelastic research . an investigation is made of the parameters to be satisfied for thermo aeroelastic similarity .  it is concluded that complete similarity obtains only when aircraft and model are identical in all respects, including size . by limiting consideration to conduction effects, by assuming the major load carrying parts of the structure are in regions where the flow is either entirely laminar, or entirely turbulent, and by assuming a specific relationship between reynolds number and nusselt number, an approach to similarity can be achieved for small scale models . experimental and analytical work is required to check on the validity of these assumptions . it appears that existing hot wind tunnels will not be completely adequ

# Evaluation Metrics (REL Ground Truth)
Compute MAP@10, NDCG@10, Precision/Recall/F1 at 10 using Cranfield REL files.

In [16]:
import os
import math
import numpy as np


rel_dir = project_dir.parent / "cranfield-dataset" / "REL"
print(rel_dir)

def load_relations(rel_folder):
    qrels = {}
    for name in os.listdir(rel_folder):
        if not name.endswith(".txt"):
            continue
        path = os.path.join(rel_folder, name)
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 3:
                    continue
                qid = int(parts[0])
                doc_id = int(parts[1])
                rel = float(parts[2])
                qrels.setdefault(qid, {})[doc_id] = rel
    return qrels

def normalize_relevance(rel_map):
    positive = [rel for rel in rel_map.values() if rel > 0]
    if not positive:
        return {}
    max_rel = max(positive)
    return {doc_id: (max_rel - rel + 1) if rel > 0 else 0.0 for doc_id, rel in rel_map.items()}

def precision_recall_f1_at_k(retrieved, relevant_set, k):
    if k <= 0:
        return 0.0, 0.0, 0.0
    top_k = retrieved[:k]
    if not top_k:
        return 0.0, 0.0, 0.0
    hit = sum(1 for doc_id in top_k if doc_id in relevant_set)
    precision = hit / k
    recall = hit / max(len(relevant_set), 1)
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

def average_precision_at_k(retrieved, relevant_set, k):
    if not relevant_set:
        return 0.0
    ap_sum = 0.0
    hit = 0
    for i, doc_id in enumerate(retrieved[:k], start=1):
        if doc_id in relevant_set:
            hit += 1
            ap_sum += hit / i
    return ap_sum / len(relevant_set)

def ndcg_at_k(retrieved, rel_map, k):
    def dcg(scores):
        return sum((2 ** rel - 1) / math.log2(i + 2) for i, rel in enumerate(scores))
    gains = [rel_map.get(doc_id, 0.0) for doc_id in retrieved[:k]]
    ideal = sorted(rel_map.values(), reverse=True)[:k]
    dcg_val = dcg(gains)
    idcg_val = dcg(ideal)
    if idcg_val == 0:
        return 0.0
    return dcg_val / idcg_val

qrels = load_relations(rel_dir)

k = 10
precision_list = []
recall_list = []
f1_list = []
ap_list = []
ndcg_list = []

for row in queries_df.to_dict(orient="records"):
    qid = int(row["id"])
    query_text = row["content"]
    hits = searcher.search(query_text, top_k=k)
    retrieved = [doc_id for doc_id, _ in hits]
    rel_map_raw = qrels.get(qid, {})
    rel_map = normalize_relevance(rel_map_raw)
    relevant_set = {doc_id for doc_id, rel in rel_map_raw.items() if rel > 0}

    precision, recall, f1 = precision_recall_f1_at_k(retrieved, relevant_set, k)
    ap = average_precision_at_k(retrieved, relevant_set, k)
    ndcg = ndcg_at_k(retrieved, rel_map, k)

    precision_list.append(precision)
    recall_list.append(recall)
    f1_list.append(f1)
    ap_list.append(ap)
    ndcg_list.append(ndcg)

metrics = {
    f"MAP@{k}": float(np.mean(ap_list)) if ap_list else 0.0,
    f"NDCG@{k}": float(np.mean(ndcg_list)) if ndcg_list else 0.0,
    f"Precision@{k}": float(np.mean(precision_list)) if precision_list else 0.0,
    f"Recall@{k}": float(np.mean(recall_list)) if recall_list else 0.0,
    f"F1@{k}": float(np.mean(f1_list)) if f1_list else 0.0,
}

metrics

C:\Itoni\CS419\final-project\cranfield-dataset\REL


{'MAP@10': 0.221915024794525,
 'NDCG@10': 0.31815700187366747,
 'Precision@10': 0.22533333333333336,
 'Recall@10': 0.37324532139496014,
 'F1@10': 0.25283663169655896}